## Preprocessing Overview

This notebook handles all data preparation before EDA and modeling. The steps below follow the same core logic as the original paper where applicable, with additional steps for ML readiness. Original/Referenced paper mentioned is Strack et al. paper.

In [1]:
import pandas as pd
import numpy as np

### Raw Data Exploration

In [2]:
#Loading the dataset
df = pd.read_csv('../data/diabetic_data.csv')
print(df.shape)
df.head()

(101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


Right away from the early inspection we can notice that weight column has '?' instead of NaN so we will have to handle this. The reference paper also confirms that question marks are used intead of missing values so we need more than just summing all nulls to see missing values.

In [3]:
# Check for missing values represented as '?'
for col in df.columns:
    count = (df[col] == '?').sum()
    if count > 0:
        percent = count / len(df) * 100 #calculate percentage of missing values
        print(f"{col}: {count} ({percent:.1f}%)")

race: 2273 (2.2%)
weight: 98569 (96.9%)
payer_code: 40256 (39.6%)
medical_specialty: 49949 (49.1%)
diag_1: 21 (0.0%)
diag_2: 358 (0.4%)
diag_3: 1423 (1.4%)


As expected we see a lot of missing values especially in weight and medical_speciality which will need to be handled properly. We also need to check for duplicate visits by same patients as those might cause data leakage and see the distribution of readmissions and non-readmissions

In [4]:
print(f"Total rows: {len(df)}")
print(f"Unique patients: {df['patient_nbr'].nunique()}")
print(f"Patients with multiple encounters: {(df['patient_nbr'].value_counts() > 1).sum()}")

Total rows: 101766
Unique patients: 71518
Patients with multiple encounters: 16773


In [5]:
print(df['readmitted'].value_counts())
print(df['readmitted'].value_counts(normalize=True).mul(100).round(1))

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64
readmitted
NO     53.9
>30    34.9
<30    11.2
Name: proportion, dtype: float64


For the purpose of this study we will use the same strategy as the original paper of bundling '>30' and 'NO' as a single case and '<30' as a seperate case effectively binarizing the variable. A pretty big class imbalance can also be observed.

We also need to check for deceased patients as deceased patients can't have readmissions. We can use the ID mapping file to see which ID's correspond to deceased patients. The relevant ID's come out to be 11, 13, 14, 19, 20, 21

In [6]:
dead_ids = [11, 13, 14, 19, 20, 21]
print(f"Rows with deceased/hospice patients: {df['discharge_disposition_id'].isin(dead_ids).sum()}")

Rows with deceased/hospice patients: 2423


Snapshot before preprocessing

This only considers NaN as null values so the actual values missing are more than this number as shown with the '?' before

In [7]:
print(f"Initial shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
df.dtypes

Initial shape: (101766, 50)
Missing values: 181168


encounter_id                int64
patient_nbr                 int64
race                          str
gender                        str
age                           str
weight                        str
admission_type_id           int64
discharge_disposition_id    int64
admission_source_id         int64
time_in_hospital            int64
payer_code                    str
medical_specialty             str
num_lab_procedures          int64
num_procedures              int64
num_medications             int64
number_outpatient           int64
number_emergency            int64
number_inpatient            int64
diag_1                        str
diag_2                        str
diag_3                        str
number_diagnoses            int64
max_glu_serum                 str
A1Cresult                     str
metformin                     str
repaglinide                   str
nateglinide                   str
chlorpropamide                str
glimepiride                   str
acetohexamide 

### Data Cleaning

In [8]:
df.replace('?', np.nan, inplace=True) #replace '?' with NaN for easier handling of missing values

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101761,443847548,100162476,AfricanAmerican,Male,[70-80),NaN,1,3,7,3,...,No,Down,No,No,No,No,No,Ch,Yes,>30
101762,443847782,74694222,AfricanAmerican,Female,[80-90),NaN,1,4,5,5,...,No,Steady,No,No,No,No,No,No,Yes,NO
101763,443854148,41088789,Caucasian,Male,[70-80),NaN,1,1,7,1,...,No,Down,No,No,No,No,No,Ch,Yes,NO
101764,443857166,31693671,Caucasian,Female,[80-90),NaN,2,3,7,10,...,No,Up,No,No,No,No,No,Ch,Yes,NO


Checking for NaN values specifically we see that two more columns `max_glu_serum` and `A1Cresult` also have nulls. They didn't show up before because we checked only for '?' before but these already had NaN's stored. However, these NaN's represent absence so we will replace these with string None signifying this category for both.

In [9]:
print(df.isnull().sum()[df.isnull().sum() > 0])

race                  2273
weight               98569
payer_code           40256
medical_specialty    49949
diag_1                  21
diag_2                 358
diag_3                1423
max_glu_serum        96420
A1Cresult            84748
dtype: int64


The paper's Table 1 defines A1Cresult as having four possible values: >8, >7, Normal, and None. Here None explicitly means "not measured." The same applies to max_glu_serum. These columns were designed by the dataset authors with None as a valid semantic category, not as a placeholder for unknown data. The NaN values in the UCI CSV version of these columns are simply cases where whoever processed the CSV left the field blank rather than writing the string None, but they mean the same thing that the test was not done. This is supported by the fact that these columns have zero ? values, meaning the original encoding did not flag them as missing in the traditional sense.

In [10]:
df['max_glu_serum'] = df['max_glu_serum'].fillna('None')
df['A1Cresult'] = df['A1Cresult'].fillna('None')

For de duplication, we will keep only the first encounter per patient. The logic is that the first visit is the cleanest representation of the patient's baseline condition before any treatment bias from prior hospital experience.

In [11]:
df = df.sort_values('encounter_id')
df = df.drop_duplicates(subset='patient_nbr', keep='first')
print(f"Rows after deduplication: {len(df)}")

Rows after deduplication: 71518


- Weight is 97% missing, imputing data would be fabricating it so we drop it
- encounter_id, patient_nbr and payer_code are just ID's and insurance codes and don't have any sigificance to the target variable so we can drop them
- We will also remove the deceased patient rows as they can't be readmitted

In [12]:
df.drop(columns=['encounter_id', 'patient_nbr', 'weight', 'payer_code'], inplace=True)

In [13]:
dead_ids = [11, 13, 14, 19, 20, 21]
df = df[~df['discharge_disposition_id'].isin(dead_ids)]
print(f"Rows after removing deceased: {len(df)}")

Rows after removing deceased: 69973


In the `specialty_known` column there are approx 49% values missing which is too much to impute meaningfully. But instead of dropping the column as done in the Strack et. al paper, we will turn it into a new binary feature. This will preserve the signal that some patients had known specialists and others didn't

In [14]:
df['specialty_known'] = df['medical_specialty'].notna().astype(int)
df.drop(columns=['medical_specialty'], inplace=True)

In [15]:
df['race'] = df['race'].fillna(df['race'].mode()[0])#only 2% missing, so we can fill with mode

`diag_1`, `diag_2`, `diag_3` each have 700+ unique ICD-9 codes. We cannot one-hot encode 700 levels as that would create 2,100 columns. Instead we will map them to 9 clinical categories exactly as the original paper did.

In [16]:
def map_diag(code):
    # Other if empty, non-numeric, or starts with V/E (external causes)
    if pd.isna(code):
        return 'Other'
    code = str(code)
    if code.startswith('V') or code.startswith('E'):
        return 'Other'
    try:
        c = float(code)
    except:
        return 'Other'
    # Map based on ICD-9 code ranges
    if 390 <= c <= 459 or c == 785:
        return 'Circulatory'
    elif 460 <= c <= 519 or c == 786:
        return 'Respiratory'
    elif 520 <= c <= 579 or c == 787:
        return 'Digestive'
    elif c == 250:
        return 'Diabetes'
    elif 800 <= c <= 999:
        return 'Injury'
    elif 710 <= c <= 739:
        return 'Musculoskeletal'
    elif 580 <= c <= 629 or c == 788:
        return 'Genitourinary'
    elif 140 <= c <= 239:
        return 'Neoplasms'
    else:
        return 'Other'

In [17]:
df['diag_1'] = df['diag_1'].apply(map_diag)
df['diag_2'] = df['diag_2'].apply(map_diag)
df['diag_3'] = df['diag_3'].apply(map_diag)

The paper states that features with near-zero variance were removed before modeling. We identify these explicitly by checking the proportion of the dominant value in each of the 24 medication columns. Any column where one value (almost always "No") accounts for 99% or more of rows carries no useful signal for the model.

In [18]:
print(df.columns.tolist())

['race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted', 'specialty_known']


Seems like the UCI CSV omits `sitagliptin` medication which appears in the paper's original Table 1. This means we have 23 medication columns not 24.

In [19]:
# List of meds to check
med_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
    'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 
    'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 
    'metformin-pioglitazone'
]

low_variance_cols = []
keep_cols = []

for col in med_cols:
    if col in df.columns:
        # Get the percentage of the most common value
        freq = df[col].value_counts(normalize=True).values[0]
        
        # If it's 99% or more the same thing, mark it for dropping
        if freq >= 0.99:
            low_variance_cols.append(col)
            print(f"Dropping {col}: {freq*100:.2f}% is the same value")
        else:
            keep_cols.append(col) #keeping track of kept columns for later use

Dropping nateglinide: 99.30% is the same value
Dropping chlorpropamide: 99.90% is the same value
Dropping acetohexamide: 100.00% is the same value
Dropping tolbutamide: 99.98% is the same value
Dropping acarbose: 99.71% is the same value
Dropping miglitol: 99.97% is the same value
Dropping troglitazone: 100.00% is the same value
Dropping tolazamide: 99.96% is the same value
Dropping examide: 100.00% is the same value
Dropping citoglipton: 100.00% is the same value
Dropping glyburide-metformin: 99.29% is the same value
Dropping glipizide-metformin: 99.99% is the same value
Dropping glimepiride-pioglitazone: 100.00% is the same value
Dropping metformin-rosiglitazone: 100.00% is the same value
Dropping metformin-pioglitazone: 100.00% is the same value


In [20]:
# dropping them from the data
df = df.drop(columns=low_variance_cols)

print(f"\nFinal list of dropped columns: {low_variance_cols}")


Final list of dropped columns: ['nateglinide', 'chlorpropamide', 'acetohexamide', 'tolbutamide', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']


In [21]:
df['readmitted'] = (df['readmitted'] == '<30').astype(int)
print(df['readmitted'].value_counts())
print(f"Readmission rate: {df['readmitted'].mean()*100:.1f}%")

readmitted
0    63696
1     6277
Name: count, dtype: int64
Readmission rate: 9.0%


The readmission rate of 9.0% shows extreme class imbalance. A naive classifier will be 91% accurate by just predicting no one will be readmitted.We will need to be careful about this while training the models and use appropriate metrics for evaluation.

`change` and `diabetesMed` are summary columns fully derivable from the individual medication columns already present in the dataset. diabetesMed = "Yes" whenever any drug column is not "No". change = "Yes" whenever any drug column is "Up" or "Down". Both are dropped to avoid redundancy. The `num_med_changes` feature can be created with feature engineering which will capture the change signal in a more informative way.

In [22]:
df.drop(columns=['change', 'diabetesMed'], inplace=True)

`admission_type_id`, `discharge_disposition_id` and `admission_source_id` are categorical codes not numbers so we convert them to strings explicitly

In [23]:
for col in ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']:
    df[col] = df[col].astype(str)

### Feature Engineering

We create a combined indicator while retaining the individual columns, as number_inpatient in particular is identified by the paper as a strong standalone predictor of readmission.

In [ ]:
df['total_prior_visits'] = (df['number_outpatient'] + df['number_emergency'] + df['number_inpatient'])

The 8 remaining medication columns each have values: No, Steady, Up, Down. We Count how many medications were actively changed (Up or Down) during this encounter. More medication adjustments in a single visit suggests the patient's condition was more unstable, which correlates with readmission risk

In [25]:
df['num_med_changes'] = df[keep_cols].isin(['Up','Down']).sum(axis=1)

`A1Cresult` has values: None, Norm, >7, >8. The original paper's central finding was that measuring HbA1c reduces readmission. We will create a binary flag for whether it was tested at all

In [26]:
df['A1C_tested'] = (df['A1Cresult'] != 'None').astype(int)

The paper (Strack et al., 2014) performed a preliminary plot of age vs readmission rate and found three distinct regions of behavior:
- [0-30): younger patients showed one readmission pattern
- [30-60): middle-aged patients showed a different pattern
- [60-100): older patients showed a third distinct pattern

Based on this observation, the paper grouped age into these 3 clinical categories rather than using the original 10-year brackets directly. We follow this exact decision. Converting to midpoint numerics (5, 15, 25...) would imply equal spacing between brackets which the paper did not assume.

In [27]:
age_map = {
    '[0-10)':  'young',
    '[10-20)': 'young',
    '[20-30)': 'young',
    '[30-40)': 'middle',
    '[40-50)': 'middle',
    '[50-60)': 'middle',
    '[60-70)': 'senior',
    '[70-80)': 'senior',
    '[80-90)': 'senior',
    '[90-100)':'senior',
}
 
df['age_group'] = df['age'].map(age_map)
df.drop(columns=['age'], inplace=True)
 
print("Age group distribution:")
print(df['age_group'].value_counts())
print(f"\nOriginal 'age' column dropped. New 'age_group' column added.")

Age group distribution:
age_group
senior    46296
middle    21869
young      1808
Name: count, dtype: int64

Original 'age' column dropped. New 'age_group' column added.


In [28]:
print(f"Final shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Readmission rate: {df['readmitted'].mean()*100:.1f}%")
df.dtypes

Final shape: (69973, 32)
Missing values: 0
Readmission rate: 9.0%


race                          str
gender                        str
admission_type_id             str
discharge_disposition_id      str
admission_source_id           str
time_in_hospital            int64
num_lab_procedures          int64
num_procedures              int64
num_medications             int64
number_outpatient           int64
number_emergency            int64
number_inpatient            int64
diag_1                        str
diag_2                        str
diag_3                        str
number_diagnoses            int64
max_glu_serum                 str
A1Cresult                     str
metformin                     str
repaglinide                   str
glimepiride                   str
glipizide                     str
glyburide                     str
pioglitazone                  str
rosiglitazone                 str
insulin                       str
readmitted                  int64
specialty_known             int64
total_prior_visits          int64
num_med_change

In [29]:
df.to_csv('../outputs/diabetes_clean.csv', index=False)
print("Saved.")

Saved.
